# NLP QLoRA

Bloc C — Dataset / QLoRA (E4 · E5).

In [ ]:
# Versions figées pour que le notebook tourne pareil chez tout le monde (règle 7).
# Seul endroit à modifier si l'équipe change une version.
# Sur Colab (relevé le 2026-10-04), seule bitsandbytes manque : les autres versions sont celles déjà installées.
PACKAGES = {
    "transformers": "5.17.0",
    "peft": "0.21.0",
    "accelerate": "1.15.0",
    "datasets": "4.8.5",
    "bitsandbytes": "0.50.2",  # nécessaire pour charger le modèle en 4-bit
}

requirements = " ".join(f"{name}=={pinned}" for name, pinned in PACKAGES.items())
%pip install -q {requirements}

# Test : les versions installées sont bien celles demandées.
from importlib.metadata import version

for name, pinned in PACKAGES.items():
    assert version(name) == pinned, f"{name} : {version(name)} installé au lieu de {pinned}"
print("Versions OK :", PACKAGES)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Repli possible : "Qwen/Qwen2.5-1.5B-Instruct" (décision prise avec la mesure de T5.1).
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"


def load_base_model(model_id: str = MODEL_ID):
    """Charge le modèle de base en 4-bit avec son tokenizer.

    Paramètres :
        model_id : identifiant du modèle sur le Hub Hugging Face.

    Retour :
        (model, tokenizer) : le modèle quantifié placé sur le GPU, et son tokenizer.

    Exemple :
        model, tokenizer = load_base_model("Qwen/Qwen2.5-1.5B-Instruct")
    """
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",  # type 4-bit prévu pour des poids répartis en cloche
        bnb_4bit_use_double_quant=True,  # quantifie aussi les constantes : économise un peu de mémoire
        bnb_4bit_compute_dtype=torch.float16,  # le T4 calcule en fp16 (décision du backlog)
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        device_map="auto",  # place le modèle sur le GPU disponible
    )
    return model, tokenizer


model, tokenizer = load_base_model()

# Test : le modèle est sur le GPU, quantifié, et le tokenizer sait formater des messages.
memory_gb = torch.cuda.memory_allocated() / 1024**3
assert next(model.parameters()).device.type == "cuda"
assert memory_gb < 4, f"{memory_gb:.1f} Go : trop pour du 4-bit"  # le 3B en fp16 dépasserait 6 Go
assert tokenizer.chat_template is not None  # nécessaire pour le format {"messages": [...]}
print(f"Modèle chargé : {MODEL_ID} | mémoire GPU : {memory_gb:.2f} Go")